# Assignment 2
## Assignment submission header
### Submission preparation instructions
Please provide the names and Drexel email addresses of all group members. If applicable, provide notes on any tutoring support received toward the completion of this submission.

### Assignment submission group
- Group member 1: Jewel Cooper (jc4895@drexel.edu)
- Group member 2: Minh Vo (mv659@drexel.edu)
- Group member 3: Quang Nguyen (qnn23@drexel.edu)

### Additional submission comments
- Tutoring support received: NA
- Other (other): NA

In [61]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Tin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

# Part 1 (25 points)

In this section, convert the comma-delimiated format provided for schedules (as shown [here](http://www3.septa.org/ccstations/me/sched_data.csv)) into a functioning schedule (as shown [here](http://www3.septa.org/ccstations/me/)). Note: Additional documentation may be found [here](http://www3.septa.org/).

__1a.__ (3 points) Update the station code to get the schedule for Suburban Station using SEPTA's API (as shown in the example in __Section 3.1.3__ of the lecture notes). Print the time at which the request was made. Print and examine the second item in `schedule`.

In [62]:
import re, requests, csv
from pprint import pprint
import datetime as dt

#---Your code starts here---
station_code = 'ss'
#---Your code stops here---
schedule_url = "".join(["http://www3.septa.org/ccstations/", station_code, "/sched_data.csv"])
response = requests.get(schedule_url)
schedule = list(csv.reader(response.text.strip().split("\n")))

print("Access time:", dt.datetime.now())
#---Your code starts here---
pprint(schedule[1])
#---Your code stops here---

Access time: 2026-05-10 23:05:49.326040
['R4S=11:25',
 'Airport',
 '4B',
 ' 1 LATE',
 'LOCAL                    ',
 '2477  ',
 '<_NEXT_MSG>11:55',
 '30th St Gray',
 '3B',
 ' 2 LATE',
 'LOCAL                    ',
 '6479  ',
 '<_NEXT_MSG>',
 '',
 '',
 '',
 '',
 '',
 '<_NEXT_MSG>',
 '',
 '',
 '',
 '',
 '',
 '']


__1b.__ (8 points) Create a list called `trains` that extracts three pieces of information from `schedule` for each train: (1) scheduled arrival time, (2) destination, and (3) status. Store this information as a list of lists. Print the first five lists in `trains`. (Note: There may be a variable number of trains for each item in `schedule`, but each train has a fixed number of attributes.)

In [63]:
trains = []

#---Your code starts here---
for row in schedule [1:]:
    if len(row) <4:
        continue
    time = row[0].split("=") [1]
    destination = row[1]
    status = row[3]
    trains.append([time,destination, status])
#---Your code stops here---

trains[:5]

[['11:25', 'Airport', ' 1 LATE'],
 ['11:35', 'Warminster', 'ON TIME'],
 ['11:09', 'Marcus Hook', ' 4 LATE'],
 ['11:50', 'West Trenton', 'ON TIME'],
 ['11:43', 'Wawa', 'ON TIME']]

__1c.__ (5 points) The arrival times are listed in 12-hour time, but they lack AM/PM information. Update `trains` by using the current system time and the fact that the schedule information contains only trains arriving in the next few hours to fix the problem (the updated arrival times should have the following format: mm/dd/yyyy hh:mm[AM/PM]). Print the first five items in the updated list. (Note: Make sure your method can handle trains schedule to arrive after midnight.)

In [64]:
from pytz import timezone

access_time = dt.datetime.now(timezone('US/Eastern'))
access_hour = access_time.hour if access_time.hour else 12
access_date = access_time.strftime("%m/%d/%Y")

if access_hour >= 12:
  access_hour -= 12
  current_am_or_pm = ["PM", "AM"]
else:
  current_am_or_pm = ["AM", "PM"]

for train in trains:
  #---Your code starts here---
  
  updated_trains = []

for train in trains:
    time_str = train[0] 
    hour = int(time_str.split(":")[0])

  
    if hour < access_hour:
        
        am_pm = "AM"
        date = access_time + dt.timedelta(days=1)
    elif access_time.hour >= 12:
        am_pm = "PM"

        date = access_time
    else:      
        am_pm = "AM"
        date = access_time

    
    formatted = f"{date.month:02d}/{date.day:02d}/{date.year} {time_str}{am_pm}"
    updated_trains.append([formatted, train[1], train[2]])

  #---Your code stops here---

    
trains[:5]

[['11:25', 'Airport', ' 1 LATE'],
 ['11:35', 'Warminster', 'ON TIME'],
 ['11:09', 'Marcus Hook', ' 4 LATE'],
 ['11:50', 'West Trenton', 'ON TIME'],
 ['11:43', 'Wawa', 'ON TIME']]

__1d.__ (4 points) Complete the `parse_time` function below, which takes a schedule such as that stored in `trains` as input and parses the arrival time data using `dateutil.parser`. The function should return a list that includes three pieces of information: (1) parsed arrival time, (2) destination, and (3) status. Prior to returning, sort the data in ascending order of arrival time. Print the first five items in the parsed data.

In [65]:
from dateutil import parser as dateparser

def parse_times(trains):

    datetime_parsed_trains = []

    #---Your code starts here---
    for train in trains:
        parsed_time = dateparser.parse(train[0])
        destination = train[1]
        status = train[2]

        datetime_parsed_trains.append([parsed_time, destination, status])

    #---Your code stops here---

    return sorted(datetime_parsed_trains, key = lambda x: x[0])

In [66]:
parsed_trains = parse_times(trains)
parsed_trains[:5]

[[datetime.datetime(2026, 5, 10, 10, 55), 'Airport', '13 LATE'],
 [datetime.datetime(2026, 5, 10, 10, 57), 'Temple U', '11 LATE'],
 [datetime.datetime(2026, 5, 10, 10, 57), 'Temple U', '10 LATE'],
 [datetime.datetime(2026, 5, 10, 11, 9), 'Marcus Hook', ' 4 LATE'],
 [datetime.datetime(2026, 5, 10, 11, 25), 'Airport', ' 1 LATE']]

__1e.__ (5 points) Complete the `save_schedule` function to create hourly log files with train information. File names should be formatted as `"data/trains/%Y-%m%-d%-H%.txt"`. Files should contain the updated arrival time, destination, and status of each train scheduled to arrive within the hour, with one train per line. Apply `save_schedule` to the parsed version of the train schedule, and print the contents of the `"data/trains/"` directory.

In [67]:
import os

def save_schedule(datetime_parsed_trains_24_hour):
    os.makedirs("data/trains/", exist_ok=True)

    #---Your code starts here---
    for train in datetime_parsed_trains_24_hour:
        train_time = train[0]
        destination = train[1]
        status = train[2]

        filename = train_time.strftime("data/trains/%Y-%m-%d-%H.txt")

        with open(filename, "a") as file:
            file.write(f"{train_time}, {destination}, {status}\n")
    #---Your code stops here---

In [68]:
save_schedule(parsed_trains)

os.listdir("data/trains/")

['.DS_Store',
 '2026-05-10-10.txt',
 '2026-05-10-11.txt',
 '2026-05-10-12.txt',
 '26-05-10-12.txt']

# Part 2 (25 points)
For this section, you will use the Sportradar Soccer v4 API to download and process the results of matches in the 2020-2021 season of the English Premier League. API documentation can be found [here.](https://api.sportradar.com/soccer/trial/v4/openapi/swagger/index.html#/seasons/getSeasons)

__2a.__ (4 points) Sign up for a Sportradar developer account and obtain an API key for the "Soccer v4" endpoints. Once you've registered an app, create a key, and update `soccer_key`. After making the API request below to request summary information from the 2022&ndash;2023 season, print the first item under "summaries" in `season_summary` and explore the data.  

In [69]:
import requests

#---Your code starts here---
soccer_key = "cDHwpAfAOnfA7RrOg1JZGPY4oby6kqgL7H5dQ2u2"
#---Your code stops here---
competition_id = "sr:season:94193"

address = "".join(["https://api.sportradar.com/soccer/trial/v4/en/seasons/", competition_id,"/summaries.json?api_key=", soccer_key])
season_summary = requests.get(address).json()
season_summary
# season_summary["summaries"][0]

{'generated_at': '2026-05-11T03:05:52+00:00', 'message': 'Wrong identifier'}

The competition_id provided ("sr:season:94193") seems to be incorrect as the data cannot be successfully fetched, with a 404 code response "Wrong Identifer". Therefore, I have chosen a different competition id: "sr:season:105353", which is the 2023-2024 of the Premier League. Because of this, the new fetched data does not contain any team with the abbreviation "SOC". A new team is chosen instead ("MCI").

In [70]:
import requests

#---Your code starts here---
soccer_key = "cDHwpAfAOnfA7RrOg1JZGPY4oby6kqgL7H5dQ2u2"
#---Your code stops here---
competition_id = "sr:season:105353"

address = "".join(["https://api.sportradar.com/soccer/trial/v4/en/seasons/", competition_id,"/summaries.json?api_key=", soccer_key])
season_summary = requests.get(address).json()

season_summary["summaries"][0]

{'sport_event': {'id': 'sr:sport_event:41762837',
  'start_time': '2023-08-11T19:00:00+00:00',
  'start_time_confirmed': True,
  'date_confirmed': True,
  'sport_event_context': {'sport': {'id': 'sr:sport:1', 'name': 'Soccer'},
   'category': {'id': 'sr:category:1',
    'name': 'England',
    'country_code': 'ENG'},
   'competition': {'id': 'sr:competition:17',
    'name': 'Premier League',
    'gender': 'men',
    'alternative_name': 'English Premier League'},
   'season': {'id': 'sr:season:105353',
    'name': 'Premier League 23/24',
    'start_date': '2023-08-11',
    'end_date': '2024-05-19',
    'year': '23/24',
    'competition_id': 'sr:competition:17'},
   'stage': {'order': 1,
    'type': 'league',
    'phase': 'regular season',
    'start_date': '2023-08-11',
    'end_date': '2024-05-19',
    'year': '23/24'},
   'round': {'number': 1},
   'groups': [{'id': 'sr:league:74627', 'name': 'Premier League 23/24'}]},
  'coverage': {'type': 'sport_event',
   'sport_event_properties': 

__2b.__ (6 points) Create a function called `get_team_results` that takes `season_summary` as input and construct a dictionary that uses abbreviated team names as keys and collects the following information for each match played by each team (`status != "postponed"`): (1) abbreviation of the opposing team, (2) goals scored, (3) goals conceded, and (4) points earned. (Note: In soccer leagues, winning a match earns 3 points for the winner and 0 points for the loser, while drawing results in 1 point for each team. The goal difference is the difference between goals for and goals against.) Store the results as `team_results`, and print the first five matches played by "SOC".

In [71]:
from collections import defaultdict

def get_team_results(season_summary):
    team_results = defaultdict(list)

    for match in season_summary["summaries"]:
        if match["sport_event_status"]['status'] != 'postponed':
            for team in match["sport_event"]["competitors"]:
        #---Your code starts here---(collect the home and away team abbreviations)
                if team['qualifier'] == 'home':
                    home_team = team['abbreviation']
                if team['qualifier'] == 'away':
                    away_team = team['abbreviation']
        #---Your code starts here---(collect the home and away scores)
            home_score = match['sport_event_status'].get('home_score', 0)
            away_score = match['sport_event_status'].get('away_score', 0)
        #---Your code stops here---

        #---Your code starts here---(collect the home and away points)
            if home_score > away_score:
                home_points = 3
                away_points = 0
            if home_score < away_score:
                home_points = 0
                away_points = 3
            else:
                home_points = 1
                away_points = 1
        #---Your code stops here---

            team_results[home_team].append({"opponent": away_team,
                                      "scored" : home_score,
                                      "conceded" : away_score,
                                      "points" : home_points})

            team_results[away_team].append({"opponent": home_team,
                                      "scored" : away_score,
                                      "conceded" : home_score,
                                      "points" : away_points})

    return team_results

In [72]:
team_results = get_team_results(season_summary)
team_results["MCI"][:5]

[{'opponent': 'BUR', 'scored': 3, 'conceded': 0, 'points': 3},
 {'opponent': 'NEW', 'scored': 1, 'conceded': 0, 'points': 1},
 {'opponent': 'SHU', 'scored': 2, 'conceded': 1, 'points': 3},
 {'opponent': 'FUL', 'scored': 5, 'conceded': 1, 'points': 1},
 {'opponent': 'WHU', 'scored': 3, 'conceded': 1, 'points': 3}]

In [73]:
team_results

defaultdict(list,
            {'BUR': [{'opponent': 'MCI',
               'scored': 0,
               'conceded': 3,
               'points': 0},
              {'opponent': 'AVL', 'scored': 1, 'conceded': 3, 'points': 0},
              {'opponent': 'TOT', 'scored': 2, 'conceded': 5, 'points': 0},
              {'opponent': 'NFO', 'scored': 1, 'conceded': 1, 'points': 1},
              {'opponent': 'MUN', 'scored': 0, 'conceded': 1, 'points': 0},
              {'opponent': 'NEW', 'scored': 0, 'conceded': 2, 'points': 1},
              {'opponent': 'LUT', 'scored': 2, 'conceded': 1, 'points': 3},
              {'opponent': 'CFC', 'scored': 1, 'conceded': 4, 'points': 0},
              {'opponent': 'BRE', 'scored': 0, 'conceded': 3, 'points': 1},
              {'opponent': 'BOU', 'scored': 1, 'conceded': 2, 'points': 1}],
             'MCI': [{'opponent': 'BUR',
               'scored': 3,
               'conceded': 0,
               'points': 3},
              {'opponent': 'NEW', 'scored

__2c.__ (5 points) Aggregate the results data for each team (key) in `team_results`, and store this information as a list called `standings`. Here, collect the abbreviated team name, the number of games played, the numbers of games won, drawn, and lost, the number of goals scored and conceded, and total points. Print the first five items in `standings`.

In [74]:
standings = []

for team in team_results:

    games_played = 0; games_won = 0; games_drawn = 0; games_lost = 0
    goals_scored = 0; goals_conceded = 0; points_earned = 0

    #---Your code starts here---
    for match in team_results[team]:
        games_played += 1
        goals_scored += match['scored']
        goals_conceded += match['conceded']
        points_earned += match['points']
        if match['points'] == 3:
            games_won += 1
        elif match['points'] == 1:
            games_drawn +=1
        else:
            games_lost += 1
    #---Your code stops here---

    standings.append([team, games_played, games_won, games_drawn, games_lost,
                      goals_scored, goals_conceded, points_earned])

standings[:5]

[['BUR', 10, 1, 4, 5, 8, 25, 7],
 ['MCI', 9, 3, 6, 0, 19, 7, 15],
 ['ARS', 10, 3, 7, 0, 23, 8, 16],
 ['NFO', 10, 1, 9, 0, 10, 15, 12],
 ['BOU', 10, 0, 7, 3, 8, 21, 7]]

__2d.__ (5 points) In the English Premier League, standings are calculated based on points earned, but let's see how points earned compares to the number of goals earned. First, sort `standings` by the number of goals scored to see which team scored the most points. Then, sort `standings` by the number of `points` to see who won the tournament. Was the team that scored the most points the same team that won the tournament?

In [75]:
#---Your code starts here---
standings.sort(key=lambda x: x[5], reverse=True)
#---Your code stops here---
print("Most goals:", standings[0])

#---Your code starts here---
standings.sort(key=lambda x: x[7])
#---Your code stops here---
print("Most points:", standings[0])

print("[No], the team that scored the most goals [is not] the same as the team who scored the most points.")

Most goals: ['NEW', 10, 1, 8, 1, 26, 11, 11]
Most points: ['SHU', 10, 0, 6, 4, 7, 29, 6]
[No], the team that scored the most goals [is not] the same as the team who scored the most points.


__2e.__ (5 points) If two teams have the same number of points, then the team with the higher difference between goals earned and goals conceded is ranked higher. If the differences are also the same, then the team with the higher number of goals scored is ranked higher. To correctly rank the teams, sort `standings` by the three factors described above. Print the sorted list.

In [76]:
#---Your code starts here---
standings.sort(key=lambda x: (x[7], x[5] - x[6], x[5]), reverse=True)
#---Your code stops here---
standings

[['TOT', 10, 4, 6, 0, 22, 9, 18],
 ['ARS', 10, 3, 7, 0, 23, 8, 16],
 ['MCI', 9, 3, 6, 0, 19, 7, 15],
 ['LFC', 10, 2, 8, 0, 23, 9, 14],
 ['AVL', 10, 2, 8, 0, 26, 14, 14],
 ['BRI', 10, 2, 7, 1, 23, 19, 13],
 ['BRE', 10, 2, 7, 1, 16, 12, 13],
 ['WHU', 10, 2, 6, 2, 16, 17, 12],
 ['WOL', 10, 2, 6, 2, 13, 17, 12],
 ['NFO', 10, 1, 9, 0, 10, 15, 12],
 ['CRY', 10, 2, 6, 2, 8, 13, 12],
 ['NEW', 10, 1, 8, 1, 26, 11, 11],
 ['CFC', 10, 2, 5, 3, 13, 11, 11],
 ['MUN', 9, 2, 5, 2, 11, 13, 11],
 ['EVE', 10, 2, 4, 4, 10, 14, 10],
 ['FUL', 10, 1, 7, 2, 9, 16, 10],
 ['LUT', 10, 1, 6, 3, 9, 20, 9],
 ['BOU', 10, 0, 7, 3, 8, 21, 7],
 ['BUR', 10, 1, 4, 5, 8, 25, 7],
 ['SHU', 10, 0, 6, 4, 7, 29, 6]]

# Part 3 (25 points)

__3a.__ (3 points) Update `file_path` and `delim` to read the text from `data/tempest.txt` and split its content into scenes. (Note: `delim` should be set to a pattern that matches the scene headers (__Section 4.4.1.2__)). Print the scenes identified in the play (e.g., "SCENE 1", "SCENE 2").

In [77]:
import re

#---Your code starts here---
file_path = "./data/tempest.txt"
delim = r"SCENE \d+\n"
#---Your code stops here---


tempest_text = open(file_path, "r").read()
scene_texts = re.split(delim, tempest_text)

print("Scenes:", scene_texts[1::2])
print(len(scene_texts))

Scenes: ["\nOn a ship at sea; a tempestuous noise of thunder and lightning\nheard\n\nEnter a SHIPMASTER and a BOATSWAIN\n\n  MASTER. Boatswain!\n  BOATSWAIN. Here, master; what cheer?\n  MASTER. Good! Speak to th' mariners; fall to't yarely, or\n    we run ourselves aground; bestir, bestir.               Exit\n\n                       Enter MARINERS\n\n  BOATSWAIN. Heigh, my hearts! cheerly, cheerly, my hearts!\n    yare, yare! Take in the topsail. Tend to th' master's\n    whistle. Blow till thou burst thy wind, if room enough.\n\n          Enter ALONSO, SEBASTIAN, ANTONIO, FERDINAND\n                     GONZALO, and OTHERS\n\n  ALONSO. Good boatswain, have care. Where's the master?\n    Play the men.\n  BOATSWAIN. I pray now, keep below.\n  ANTONIO. Where is the master, boson?\n  BOATSWAIN. Do you not hear him? You mar our labour;\n    keep your cabins; you do assist the storm.\n  GONZALO. Nay, good, be patient.\n  BOATSWAIN. When the sea is. Hence! What cares these\n    roarers for

__3b.__ (6 points) Splitting the text on scene fails to account for act information. Update `delim` to capture the structure of both acts and scenes. Print the acts and scenes identified in the play (e.g., "ACT I. SCENE 1").

In [78]:
#---Your code starts here---
delim = r"(ACT [A-Z]+\. SCENE \d+|SCENE \d+)\n"
#---Your code stops here---

act_scene_texts = re.split(delim, tempest_text)

print("Acts and scenes:", act_scene_texts[1::2])

Acts and scenes: ['ACT I. SCENE 1', 'SCENE 2', 'ACT II. SCENE 1', 'SCENE 2', 'ACT III. SCENE 1', 'SCENE 2', 'SCENE 3', 'ACT IV. SCENE 1', 'ACT V. SCENE 1']


__3c.__ (2 points) Examine the text from `ACT. I. SCENE 1`. Is `\n` a sufficient deliminator for splitting the text by speaker? Why or why not?

In [79]:
print("No because some dialogues are more than just a line for some characters")

No because some dialogues are more than just a line for some characters


__3d.__ (7 points) Complete the `process_scenes` function to separate lines in `scene_text` by `\n`, separate speakers from the beginnings of their speeches (using regex), and collect the multiline speeches in a single string. Each speaker-line pair should be stored in the `defaultdict` structure set up in the code below. Store each speaker-line pair as a list. Process `act_scene_texts` with `process_scenes`, and print the first five lines of `ACT I`, `SCENE 1`.

In [80]:
from collections import defaultdict

def process_scenes(act_scene_texts):
  # for i, item in enumerate(act_scene_texts):
  #   print(f"[{i}]: {repr(item[:80])}")
  data = defaultdict(lambda : defaultdict(list))
  current_act, current_scene = "", ""

  ## set dictionary keys
  for act_scene, scene_text in zip(act_scene_texts[1::2], act_scene_texts[2::2]):
    if act_scene[:3] == "ACT": # if it's a new act, update the current_act value
      #---your code starts here--- (separate and collect the act/scene information) 
      current_act, current_scene = act_scene.split(". ")
      #---your code stops here---
    else:# otherwise assign the act_scene_heading
      #---your code sarts here--- (just update the current_scene)
      current_scene = act_scene
      #---your code stops here---


    speaker, speech = "", ""

    for line in scene_text.split("\n"):
      #---Your code starts here---
      match = re.search(r"^\s+([A-Z]{3,})\.\s", line)
      if match:
        if speaker and speech:
            data[current_act][current_scene].append((speaker, speech.strip()))
        speaker = match.group(1)
        speech = re.sub(r"^\s*[A-Z]{3,}\.\s*", "", line).strip()
      elif speaker and line.strip():
          speech += " " + line.strip()
      # print("Line:", speech)

    if speaker and speech:
      data[current_act][current_scene].append((speaker, speech.strip()))
      # print(dict(data))

      # ---Your code stops here---

  return data

data = process_scenes(act_scene_texts)
data['ACT I']['SCENE 1'][:5]

[('MASTER', 'Boatswain!'),
 ('BOATSWAIN', 'Here, master; what cheer?'),
 ('MASTER',
  "Good! Speak to th' mariners; fall to't yarely, or we run ourselves aground; bestir, bestir.               Exit Enter MARINERS"),
 ('BOATSWAIN',
  "Heigh, my hearts! cheerly, cheerly, my hearts! yare, yare! Take in the topsail. Tend to th' master's whistle. Blow till thou burst thy wind, if room enough. Enter ALONSO, SEBASTIAN, ANTONIO, FERDINAND GONZALO, and OTHERS"),
 ('ALONSO', "Good boatswain, have care. Where's the master? Play the men.")]

In [81]:
data = process_scenes(act_scene_texts)
data['ACT I']['SCENE 1'][:5]

[('MASTER', 'Boatswain!'),
 ('BOATSWAIN', 'Here, master; what cheer?'),
 ('MASTER',
  "Good! Speak to th' mariners; fall to't yarely, or we run ourselves aground; bestir, bestir.               Exit Enter MARINERS"),
 ('BOATSWAIN',
  "Heigh, my hearts! cheerly, cheerly, my hearts! yare, yare! Take in the topsail. Tend to th' master's whistle. Blow till thou burst thy wind, if room enough. Enter ALONSO, SEBASTIAN, ANTONIO, FERDINAND GONZALO, and OTHERS"),
 ('ALONSO', "Good boatswain, have care. Where's the master? Play the men.")]

__3e.__ (5 points) Count the number of lines spoken by each speaker in the play (across every `'ACT'` and `'SCENE'`). Print the top five speakers by number of lines spoken.

In [82]:
from collections import Counter

#---Your code starts here---
speech_counts = Counter()

for act, scenes in data.items():
    for scene, lines in scenes.items():
        for speaker, speech in lines:
            speech_counts[speaker] += 1
#---Your code stops here---

speech_counts.most_common(5)

[('PROSPERO', 113),
 ('SEBASTIAN', 67),
 ('STEPHANO', 60),
 ('ANTONIO', 59),
 ('GONZALO', 52)]

__3f.__ (2 points) Comment on any limitations of the process developed in this section. Do you see any artifacts indicating imprecision?

In [83]:
print("The developed process is limited in terms of speech detection accuracy. This limitation is evidenced by stage directions being captured as speech ('Exit CALIBAN', 'Enter ARIEL', etc.)")

The developed process is limited in terms of speech detection accuracy. This limitation is evidenced by stage directions being captured as speech ('Exit CALIBAN', 'Enter ARIEL', etc.)


# Part 4 (25 points)

In this section, you will work with a subset of the [Lending Club Loan Dataset](https://www.kaggle.com/wordsforthewise/lending-club) located in the data folder (./data/loan_extra-small.csv).

__4a.__ (3 points) Update the file path. How many fields are included in `loan_data`? How many rows are included in `load_data`?

In [84]:
import csv

#---Your code starts here---
file_path = './data/loan_extra-small.csv'
with open(file_path, 'r') as f:
    reader = csv.reader(f)
    fields = next(reader)
    loan_data = list(reader)

#---Your code stops here---

print("Number of fields: ", len(fields))
print("Number of rows:", len(loan_data))

Number of fields:  145
Number of rows: 49999


__4b.__ (2 points) Create a dictionary called `statuses` whose keys are the unique items in the `loan_status` field and whose values are set to `1` for a `loan_status` of `"Current"` or `"Fully Paid"` and `0` otherwise. Print the dictionary.

In [85]:
statuses = {}

#---Your code starts here---
status_index = fields.index("loan_status") 
for row in loan_data:
  status = row[status_index]
  statuses[status] = 1 if status in ("Current", "Fully Paid") else 0
#---Your code stops here---

statuses

{'Current': 1,
 'Fully Paid': 1,
 'Charged Off': 0,
 'Late (16-30 days)': 0,
 'Late (31-120 days)': 0}

__4c.__ (6 points) Count the words used in the loan descriptions (in the `desc` field) for each boolean status assigned in `statuses`. To do so, tokenize the (lowercased) description (using `word_tokenize` from `nltk.tokenize`) and then create two `Counter()` data structures that count the words in the descriptions for loans with statuses of `0` and `1`, respectively. Print the number of times the word 'loan' occurs in each set of descriptions.

In [86]:
from nltk.tokenize import word_tokenize
from collections import Counter, defaultdict

counts = defaultdict(Counter)

#---Your code starts here---
desc_index = fields.index("desc")
for row in loan_data:
  lower = row[desc_index].lower()
  tokens = word_tokenize(lower)          
  status = statuses[row[status_index]]  
  counts[status].update(tokens) 

#---Your code stops here---

print("Number of times 'loan' occurs in descriptions of loans with a status of 0:", counts[0]["loan"])
print("Number of times 'loan' occurs in descriptions of loans with a status of 1:", counts[1]["loan"])

Number of times 'loan' occurs in descriptions of loans with a status of 0: 3240
Number of times 'loan' occurs in descriptions of loans with a status of 1: 19431


__4d.__ (5 points) From the counters above, print the top 25 words for loans of each boolean status. How many words are in the top 25 for both status types?

In [87]:
#---Your code starts here---
top25_0 = set(word for word, _ in counts[0].most_common(25))
top25_1 = set(word for word, _ in counts[1].most_common(25))

print("Top 25 words for status 0:", top25_0)
print("Top 25 words for status 1:", top25_1)

in_both = top25_0 & top25_1
print("Words in top 25 for both:", in_both)
print("Number of words in top 25 for both:", len(in_both))
#---Your code stops here---


Top 25 words for status 0: {'loan', '.', '>', 'to', 'borrower', 'have', 'and', 'my', 'on', 'pay', 'br', 'cards', 'credit', 'of', ',', 'a', 'debt', 'added', 'the', 'card', 'this', 'off', 'for', '<', 'i'}
Top 25 words for status 1: {'loan', '.', '>', 'to', 'borrower', 'have', 'and', 'my', 'on', 'pay', 'br', 'credit', 'cards', 'of', ',', 'a', 'debt', 'added', 'the', 'card', 'this', 'off', 'for', '<', 'i'}
Words in top 25 for both: {'loan', '.', '>', 'to', 'borrower', 'have', 'and', 'my', 'on', 'pay', 'br', 'cards', 'credit', 'of', ',', 'a', 'debt', 'added', 'the', 'card', 'this', 'off', 'for', '<', 'i'}
Number of words in top 25 for both: 25


__4e.__ (5 points) Determine which words are unique to each status type by creating two lists: one of words that occur in descriptions of loans with statuses of `0` but not in those of loans with statuses of `1`, and one for which the opposite is true. Print the number of words in each list.

In [88]:
#---Your code starts here---
words_0 = set(counts[0].keys())
words_1 = set(counts[1].keys())

unique_to_0 = words_0 - words_1  # in 0 but not 1
unique_to_1 = words_1 - words_0  # in 1 but not 0

print("Words unique to status 0:", len(unique_to_0))
print("Words unique to status 1:", len(unique_to_1))
#---Your code stops here---

Words unique to status 0: 1763
Words unique to status 1: 12785


__4f.__ (4 points) Print the top 25 unique words for loans of each boolean status. Do these sets of words indicate anything interesting about the differences between loan statuses?

In [89]:
#---Your code starts here---
unique_counts_0 = Counter({w: c for w, c in counts[0].items() if w in unique_to_0})
unique_counts_1 = Counter({w: c for w, c in counts[1].items() if w in unique_to_1})

top25_unique_0 = unique_counts_0.most_common(25)
top25_unique_1 = unique_counts_1.most_common(25)

print("Top 25 unique words for status 0:", top25_unique_0)
print("Top 25 unique words for status 1:", top25_unique_1)
#---Your code stops here---

print("\n These descriptions show that status 0 tend to use words related to financial hardship, while status 1 use words related to investment and home improvement.")

Top 25 unique words for status 0: [('ailing', 4), ('aarp', 3), ('tent', 3), ('moped', 3), ('payoffcreditcards', 3), ('circumtance', 3), ('medicare', 2), ('doughnut', 2), ('5000.00.', 2), ('460.00', 2), ('shipments', 2), ('honey', 2), ('exxonmobil', 2), ('gratefully', 2), ('reorganization.', 2), ('dang', 2), ('600/mo', 2), ('150.00.', 2), ('pulls', 2), ('deny', 2), ('finanical', 2), ('rifinancing', 2), ('partnership.', 2), ('md', 2), ('shops', 2)]
Top 25 unique words for status 1: [('rsquo', 59), ('~', 44), ('rifle', 40), ('favorable', 33), ('1200', 28), ('broken', 27), ('dog', 26), ('gave', 26), ('11/16/13', 26), ('repairing', 25), ('roofing', 24), ('1,000', 23), ('750', 23), ('11/09/13', 23), ('relatively', 22), ('patrol', 22), ('atv', 21), ('liner', 21), ('daniel', 21), ('defense', 21), ('university', 20), ('investors.', 20), ('machine', 20), ('honda', 20), ('lock', 20)]

 These descriptions show that status 0 tend to use words related to financial hardship, while status 1 use words 